# Person 1 (Aarya): run the pipelines on a Colab GPU

**Before running anything:** menu **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 is the free one; L4/A100 if you have Colab Pro) → Save.

Then run the cells **from top to bottom** (click a cell, press **Shift + Enter**). Each cell says what it does and what you should see.

If Colab disconnects, just run all cells again from the top: finished questions are skipped, nothing is lost *as long as you saved results* (step 7).

## 1. Check the GPU
You should see a table with a GPU name (Tesla T4, L4 or A100). If it says *command not found*, the runtime has no GPU: redo the menu step above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Download the project from GitHub
Re-running this cell later just pulls the newest version of `main`.

In [ ]:
import os
REPO = "/content/LateComers-CSE576-NLP"
if not os.path.exists(REPO):
    !git clone https://github.com/aaryapendharkar/LateComers-CSE576-NLP.git {REPO}
%cd {REPO}
!git pull -q origin main
!git log --oneline -1

## 3. Install the packages (2–4 minutes)
Ignore red *dependency conflict* warnings unless the cell ends with an error.

In [ ]:
!pip install -q -r requirements-core.txt
!pip install -q -U transformers accelerate sentence-transformers
import transformers; print("transformers", transformers.__version__)

## 4. Pick the model that fits this GPU
Writes `configs/local.yaml` (only on this Colab machine; never uploaded).

The planned Qwen3.5-9B is not usable here: its new layer type needs extra GPU kernels, and without them **one question took ~5 minutes** on a T4 (Nov 13 test). We use **Qwen3-VL**, the same Qwen family with standard layers:
- T4 / L4 (≤ 30 GB): **Qwen/Qwen3-VL-4B-Instruct**
- A100 (Colab Pro): **Qwen/Qwen3-VL-8B-Instruct**

This is recorded in the report under *Changes*.

In [ ]:
import os, torch, yaml
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory fragmentation
gpu = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
big = mem_gb >= 30
model = "Qwen/Qwen3-VL-8B-Instruct" if big else "Qwen/Qwen3-VL-4B-Instruct"
local = {
    "pipelines": {"backend": "real", "p1_text_source": "standin"},
    "models": {
        "generator_fallback": model,
        "reranker": None,
        "device": "cuda",
        "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16",
        "batch_size": 8 if big else 4,
    },
    # T4 test (Nov 13): 5 pages x 4,000 chars ran out of memory on every question; 2,000 always fits.
    # Tables are never cut, only the prose around them.
    "generation": {"max_page_chars": 4000 if big else 2000,
                   "max_image_pixels": 600000 if big else 300000},
}
with open("configs/local.yaml", "w") as f:
    yaml.safe_dump(local, f)
print(f"GPU: {gpu} ({mem_gb:.0f} GB)  ->  generator: {model}, dtype: {local['models']['dtype']}, "
      f"page text cap: {local['generation']['max_page_chars']} chars")

## 5. The model test: 5 questions through P1 (text pipeline)
First run downloads the models (BGE-M3 ≈ 2 GB, Qwen3-VL-4B ≈ 9 GB) and embeds all 500 pages once, so it takes **5–10 minutes**. After that, each question should take **seconds, not minutes**.

In [ ]:
!rm -f results/runs/P1_R2_finqa.jsonl   # start the test fresh
!python scripts/run_baselines.py --pipelines P1 --runs R2 --limit 5

## 6. Look at the answers
**What good looks like:**
- `answer` is a number (or yes/no) for every row, not empty
- `cited` lists page ids
- `gold_rank` is mostly 1–5 (retrieval found the right page)
- `correct` is True for at least some rows (a few wrong answers are normal)

Copy this table into the chat with Claude.

In [ ]:
import pandas as pd, json
rows = [json.loads(l) for l in open("results/runs/P1_R2_finqa.jsonl")]
ex = {json.loads(l)["qid"]: json.loads(l) for l in open("data/examples.jsonl")}
df = pd.DataFrame([{
    "qid": r["qid"], "gold": ex[r["qid"]]["gold_answer"][:10], "answer": r["answer"],
    "correct": r["correct"], "gold_rank": r["gold_rank_retr"], "cited": r["cited"],
    "secs": r["latency_s"], "raw_output_start": r["answer_raw"][:80],
} for r in rows])
pd.set_option("display.max_colwidth", 80)
df

## 7. Save results to your Google Drive
Colab deletes everything when it disconnects. This copies `results/` to **My Drive / cse576_results** (it will ask permission to access Drive the first time).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/cse576_results
!cp -r results/. /content/drive/MyDrive/cse576_results/
!ls /content/drive/MyDrive/cse576_results/runs

---
## 8. FULL P1 baselines: 200 questions × R1, R2, R3 (≈ 1–2 hours)
Run this only after step 6 looks good. It uses the page **text** straight from `data/docs.jsonl`, so it does not need Aaron's rendered pages. (When his pages arrive, we re-run P1 on the real PDF text and run P2 on the images.)

**If Colab disconnects:** run cells 1–4, 7 (mount Drive) and the *restore* cell below, then this cell again; finished questions are skipped.

In [ ]:
# restore earlier progress from Drive (safe to run even the first time)
import os
if os.path.exists("/content/drive/MyDrive/cse576_results/runs"):
    !mkdir -p results/runs && cp -n /content/drive/MyDrive/cse576_results/runs/*.jsonl results/runs/ 2>/dev/null; ls results/runs

In [ ]:
!rm -f results/runs/P1_R2_finqa.jsonl   # drop the 5-question test so R2 is one clean run
!python scripts/run_baselines.py --pipelines P1 --runs R1 R2 R3
!mkdir -p /content/drive/MyDrive/cse576_results && cp -r results/. /content/drive/MyDrive/cse576_results/
!cat results/baselines/table1_finqa.md

## 9. Download the results to upload to GitHub
Downloads `p1_results.zip`. Unzip it **into your LateComers-CSE576-NLP folder** on your laptop (choose *Replace*), then commit and push from PowerShell as usual.

In [ ]:
!rm -f p1_results.zip && zip -qr p1_results.zip results/runs results/r2_correct.json results/baselines
from google.colab import files
files.download("p1_results.zip")

---
## 10. Gap 1 attacks on P1 (Saturday, ≈ 1 hour)
**Before this:** Janhavi's pull request is merged into `main` **and** her `data/attacks/g1.jsonl` is on `main`.
Run cells 1–4, the 2,000-character cell, and 7 first (fresh Colab session), then these.

No rendered pages needed: for P1 the runner builds each poisoned page by applying Janhavi's edit to the document text, exactly as it builds clean pages.

In [ ]:
!git pull -q origin main
!wc -l data/attacks/g1.jsonl
!mkdir -p results/runs && cp -n /content/drive/MyDrive/cse576_results/runs/*.jsonl results/runs/ 2>/dev/null
!cp -n /content/drive/MyDrive/cse576_results/r2_correct.json results/ 2>/dev/null; ls results

In [ ]:
!python scripts/run_attacks.py --pipeline P1 --gap G1 --run-name G1
!mkdir -p results/gap1 && python scripts/summarize.py --runs results/runs/P1_G1.jsonl --out results/gap1/funnel_P1.csv
!cp -r results/. /content/drive/MyDrive/cse576_results/

Then run **step 9** again to download the results and push them to GitHub. Paste the funnel table printed above into the chat.

---
## Later steps
P2 (images) and Gap 2 need Aaron's renderer and rendered pages on `main`. Claude will add those cells when they exist.